# LIBERO **eval** — 노드 C (GPU 4장)

**이미 학습된 체크포인트**로 LIBERO-10 을 평가한다(학습 안 함).
노드 3대(**A=2 · B=4 · C=4** GPU, 합 10)로 eval 부하를 나눈다 — 세 노드에서 각자 `eval_node{A,B,C}` 실행.

- **탐색 → 분배**: 실제로 체크포인트가 있는 `(모델,seed)` 만 찾아서 GPU 비(2:4:4)로 나눈 뒤 이 노드 몫만 평가.
  → seed 가 3개든 4개든, 빠진 모델이 있든 **있는 것만** 자동으로 맞춘다.
- **150k 체크포인트 × 500ep**, 외란 없음. action(.pt) 기록 → jerk/LDJ/SPARC/SignFlip 측정.
- ⚠️ **LIBERO 시뮬 필요**(robosuite/libero). 동시 env = batch×10 → batch 작게(`LIBERO_EVAL_BATCH`).
- ⚠️ 노드 간 **공유 FS 가정**(한 노드에서 모든 체크포인트가 보여야 분배가 일관됨 — 클러스터 표준).
- 결과 = `eval_clean/libero_10/<model>/seed<N>/rep0/` → `reports/09_report_sr`·`utils/0l_collect_libero` 자동 pooled.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

## 1) 학습된 체크포인트 탐색 + 이 노드 몫
먼저 **전 노드가 보는** 학습된 (모델,seed) 를 확인하고(세 노드에서 같아야 함), 이 노드가 맡을 것만 추린다.


In [ ]:
NODE = 'C'
N_EP = 500                              # 은지 요청 = 500ep
STEP = cf.CKPT_STEP                      # 150k 체크포인트

all_pairs = cf.libero_trained_pairs(step=STEP)      # 실제 체크포인트 있는 것만
mine = cf.node_split(all_pairs)[NODE]               # 이 노드 몫 (2:4:4)
gpus = cf.available_gpus()[:cf.NODE_GPUS[NODE]]
print(f'학습된 (모델,seed): {len(all_pairs)}개  (세 노드에서 이 숫자가 같아야 함)')
print(f'노드 {NODE} | GPU {gpus} | 이 노드 {len(mine)}개 × {N_EP}ep\n')
for t, s in mine:
    print(f'   {t:10} seed{s}')
if not all_pairs:
    print('\n⚠️ 학습된 libero_10 체크포인트를 못 찾음 — 경로/step 확인 (cf.OUTPUT_BASE)')

## 2) 실행 — 500ep, GPU 하나당 eval 하나(OOM 방지, 이미 끝난 건 skip)


In [ ]:
cf.run_libero_eval_jobs(mine, gpus=gpus, n_episodes=N_EP, step=STEP)

## 3) 결과 (SR + 저장 영상 수)


In [ ]:
import glob
print(f"{'MODEL':<12}{'SEED':>5}{'SR':>9}{'VIDEOS':>8}")
print('-' * 34)
for t, s in mine:
    st = cf.get_eval_status(t, s, 'libero_10')
    sr = f"{st['sr']*100:.1f}%" if st['sr'] is not None else '-'
    vids = glob.glob(str(cf.eval_clean_dir(t, s, 'libero_10') / '**' / '*.mp4'), recursive=True)
    print(f'{t:<12}{s:>5}{sr:>9}{len(vids):>8}')